# **Cross-Family LLM Stitching: Comparing Reconstruction and Prediction-Based Objectives**

*COMS 4705 - Natural Language Processing - Final Project*

### **Authors**

Caroline Zhang (cz2977@columbia.edu)

Siddharth Grover (sg4792@columbia.edu)

Edwin Argudo (eaa2193@columbia.edu)

### **GitHub Repo**

https://github.com/edwin-a14/coms-4705-project-cross-model-stitching


## **Milestone 1**

**Prerequisite:** Run the **entire [smoke-test.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/smoke-test.ipynb) successfully**
before using this notebook to verify the environment, model access, activation extraction,
output-head reconstruction, cache loading, and the small affine MSE example.

This notebook organizes the data preparation, cached representations, baseline comparisons,
and validation protocol for cross-family LLM stitching. The experiment maps Llama-3.2-3B's
final residual representation to Qwen2.5-3B's representation space using an affine transformation.
Milestone 1 establishes the data and evaluation framework for comparing reconstruction and
prediction objectives in Milestone 2.

Dataset construction is defined in [dataset.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/dataset.ipynb), and activation extraction
is defined in [activation.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/activation.ipynb). This notebook uses their saved artifacts
for analysis, with the selected runs identified by manifest checksums and model/tokenizer revisions.

### **Deliverables**

| Phase | Deliverable |
|---|---|
| 1. LAMBADA preprocessing and manifests | Reproducible example manifests, aligned labels, and a documented sampling protocol |
| 2. Sequential activation extraction and persistent caching | Paired activation caches, output-component specifications, and extraction metadata |
| 3. Native and constant-activation baselines | Native Llama, native Qwen, and train-mean Qwen baseline comparisons with sample counts |
| 4. Pipeline validation | Data alignment, split separation, reconstruction, and numerical consistency checks |
| 5. Prep for MSE/CE/KL experiments | Documented cache interface, baseline summary, and an untrained affine scaffold |

**Evaluation scope:** The primary task is prediction of the first continuation token. Full-word
LAMBADA evaluation requires a separate protocol. Llama's token-level scores are reported in its
own vocabulary, while the constant baseline and stitched-model evaluation use Qwen's vocabulary.
Analysis in this notebook focuses on training and validation; test scores are reserved for final evaluation.


## **Dependencies & Configuration**


In [1]:
%pip install -q \
    "transformers>=5,<6" \
    "huggingface_hub>=1.32,<2" \
    "datasets>=3,<5" \
    "accelerate>=1.0" \
    safetensors \
    tqdm \
    "nltk>=3.9.1,<4" \
    numpy \
    matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/coms-4705-project')
TRAIN_DIR = Path(
    "/content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0"
)
CACHE_RUN_DIR = Path(
    "/content/drive/MyDrive/coms-4705-project/activations/lambada/"
    "7b6847a7f51353d0/train10000_validation4869_a938da615918"
)
ANALYSIS_SPLITS = ('train', 'validation')

## **Phase 1 — LAMBADA Preprocessing and Manifests**

**Implementation:** [dataset.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/dataset.ipynb)

LAMBADA provides training novels and separate validation/test passages. Training examples are
sampled from the novels as context-and-next-word pairs, with passage lengths drawn from the
validation length distribution. The sampler selects noun targets that also occur earlier in
the context, normalizes training whitespace to single spaces, and distributes sampling across
novels. These criteria define the training distribution used for the stitching experiment.

Each example uses the same text prefix for both models and must fit within 256 tokens under
both tokenizers. Labels are derived by tokenizing the complete prefix and continuation, checking
that the prefix token IDs remain unchanged, and selecting the first continuation token. Multi-token
targets are retained and identified separately for subset evaluation.

The training manifest stores example IDs, source locations, prefixes, continuations, and model-specific
labels. Its accompanying provenance file records the dataset/model revisions, sampling settings,
and manifest checksum. This section characterizes the selected data through sample counts,
passage and token lengths and target characteristics. Split comparisons in Phase 2
cover the full training manifest and the selected validation cache manifest.


In [4]:
import hashlib
import json
from collections import Counter
from itertools import combinations
from IPython.display import Markdown, display


def show_table(title, columns, rows):
    def format_value(value):
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = [f"### {title}", "", "| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(map(format_value, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def read_manifest(path, expected_sha256):
    data = Path(path).read_bytes()
    if sha256_bytes(data) != expected_sha256:
        raise ValueError(f"Manifest checksum mismatch: {path}")
    rows = []
    for line_number, line in enumerate(data.splitlines(), 1):
        if not line.strip():
            raise ValueError(f"Blank manifest row at {path}:{line_number}")
        row = json.loads(line)
        if not isinstance(row, dict):
            raise ValueError(f"Expected a JSON object at {path}:{line_number}")
        rows.append(row)
    if not rows:
        raise ValueError(f"Empty manifest: {path}")
    return rows


def normalized_context(row):
    return " ".join(row["prefix"].split())


def validate_manifest(rows, split, max_tokens):
    ids, contexts = set(), set()
    for row in rows:
        if row.get("split") != split or not isinstance(row.get("id"), str):
            raise ValueError(f"Invalid split/ID in {split} manifest")
        for key in ("prefix", "continuation"):
            if not isinstance(row.get(key), str) or not row[key].strip():
                raise ValueError(f"Invalid {key} for {row['id']}")
        if row["id"] in ids:
            raise ValueError(f"Duplicate ID in {split}: {row['id']}")

        if split == "train" and normalized_context(row) in contexts:
            raise ValueError(
                f"Duplicate normalized training context: {row['id']}"
            )
        ids.add(row["id"])
        contexts.add(normalized_context(row))
        if row.get("n_words") != len((row["prefix"] + row["continuation"]).split()):
            raise ValueError(f"Word count mismatch for {row['id']}")
        for name in ("llama", "qwen"):
            for suffix in ("length", "target_id", "target_length"):
                value = row.get(f"{name}_{suffix}")
                if type(value) is not int or value < (0 if suffix == "target_id" else 1):
                    raise ValueError(f"Invalid {name}_{suffix} for {row['id']}")
            if row[f"{name}_length"] > max_tokens:
                raise ValueError(f"Context cap exceeded for {row['id']}")


def load_training_artifacts(directory):
    if directory is None:
        raise ValueError("Set TRAIN_DIR to the saved training directory printed by dataset.ipynb.")
    directory = Path(directory)
    meta = json.loads((directory / "provenance.json").read_text())
    if meta.get("split") != "train" or set(meta.get("models", {})) != {"llama", "qwen"}:
        raise ValueError("Expected training provenance for Llama and Qwen")
    if set(meta.get("revisions", {})) != {"llama", "qwen"}:
        raise ValueError("Missing tokenizer/model revisions")
    rows = read_manifest(directory / "examples.jsonl", meta["manifest_sha256"])
    if len(rows) != meta["n_train"]:
        raise ValueError("Training row count differs from provenance")
    validate_manifest(rows, "train", meta["max_tokens"])
    for row in rows:
        if len(row["id"].split("/")) != 4 or not row["id"].startswith("train/"):
            raise ValueError("Expected train/<novel>/<start>/<prefix_end> source IDs")
        if row.get("target_pos") not in meta["target_pos_tags"]:
            raise ValueError(f"Target POS violates saved sampling policy: {row['id']}")
        if row["continuation"].strip().casefold() not in {word.casefold() for word in row["prefix"].split()}:
            raise ValueError(f"Training target does not occur in its prefix: {row['id']}")
    return rows, meta


def describe_lengths(values):
    values = sorted(values)
    def percentile(q):
        return values[round(q * (len(values) - 1))]
    return [len(values), min(values), sum(values)/len(values), percentile(.5), percentile(.95), max(values)]


training_rows, training_meta = load_training_artifacts(TRAIN_DIR)
training_by_id = {row["id"]: row for row in training_rows}
dataset_summary = {
    "examples": len(training_rows),
    "source_novels": len({row["id"].split("/")[1] for row in training_rows}),
    "manifest_sha256": training_meta["manifest_sha256"],
    "target_pos_counts": dict(Counter(row["target_pos"] for row in training_rows)),
}
show_table("Training data", ["Measure", "Value"], [
    ["Examples", dataset_summary["examples"]], ["Source novels", dataset_summary["source_novels"]],
    ["Dataset", training_meta["dataset"]], ["Seed", training_meta["seed"]],
    ["Context cap (tokens/model)", training_meta["max_tokens"]],
    ["Target POS tags", ", ".join(training_meta["target_pos_tags"])],
    ["Target occurrence rule", training_meta["target_in_prefix"]],
    ["Whitespace policy", training_meta["whitespace"]], ["Sampling policy", training_meta["sampling"]],
])
show_table("Passage and context lengths", ["Measure", "N", "Min", "Mean", "Median", "P95", "Max"],
           [[label, *describe_lengths(row[field] for row in training_rows)] for label, field in
            [("Passage words (including target)", "n_words"), ("Llama prefix tokens", "llama_length"),
             ("Qwen prefix tokens", "qwen_length")]])
show_table("Target tokenization", ["Model", "Single-token targets", "All targets", "Single-token fraction"],
           [[name, sum(row[f"{name}_target_length"] == 1 for row in training_rows), len(training_rows),
             sum(row[f"{name}_target_length"] == 1 for row in training_rows)/len(training_rows)]
            for name in ("llama", "qwen")])
show_table("Target POS distribution", ["Tag", "Examples"], sorted(dataset_summary["target_pos_counts"].items()))
show_table("Artifact provenance", ["Field", "Value"], [
    ["Training directory", Path(TRAIN_DIR)], ["Manifest SHA256", training_meta["manifest_sha256"]],
    ["Dataset revision", training_meta["dataset_revision"]],
    *[[f"{name} model", training_meta["models"][name]] for name in ("llama", "qwen")],
    *[[f"{name} revision", training_meta["revisions"][name]] for name in ("llama", "qwen")],
])


### Training data

| Measure | Value |
| --- | --- |
| Examples | 10000 |
| Source novels | 2500 |
| Dataset | cimec/lambada |
| Seed | 42 |
| Context cap (tokens/model) | 256 |
| Target POS tags | NN, NNP, NNPS, NNS |
| Target occurrence rule | case-insensitive whitespace-delimited word |
| Whitespace policy | single spaces |
| Sampling policy | balanced first pass with productive-novel refill |

### Passage and context lengths

| Measure | N | Min | Mean | Median | P95 | Max |
| --- | --- | --- | --- | --- | --- | --- |
| Passage words (including target) | 10000 | 59 | 74.974 | 73 | 95 | 170 |
| Llama prefix tokens | 10000 | 59 | 81.627 | 79 | 105 | 195 |
| Qwen prefix tokens | 10000 | 58 | 80.715 | 78 | 104 | 194 |

### Target tokenization

| Model | Single-token targets | All targets | Single-token fraction |
| --- | --- | --- | --- |
| llama | 7740 | 10000 | 0.774 |
| qwen | 7728 | 10000 | 0.773 |

### Target POS distribution

| Tag | Examples |
| --- | --- |
| NN | 8917 |
| NNP | 7 |
| NNPS | 1 |
| NNS | 1075 |

### Artifact provenance

| Field | Value |
| --- | --- |
| Training directory | /content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0 |
| Manifest SHA256 | 604387bf620dfe540501006eb7977fa73c99a31454f9e6093c0cc497747d3044 |
| Dataset revision | 5953bd97664b64b95754f299b2309ecfbfbe81b9 |
| llama model | meta-llama/Llama-3.2-3B |
| qwen model | Qwen/Qwen2.5-3B |
| llama revision | 13afe5124825b4f3751f836b40dafda64c1ed062 |
| qwen revision | 3aab1f1954e9cc14eb9509a215f9e5ca08227a9b |

## **Phase 2 — Sequential Activation Extraction and Persistent Caching**

**Implementation:** [activation.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/activation.ipynb), [activation-full.ipynb](https://github.com/edwin-a14/coms-4705-project-cross-model-stitching/blob/306fe3c3fdd168b53780c51a42c786d304c991c8/notebooks/activation-full.ipynb)

Activation extraction pairs representations of identical text contexts from Llama and Qwen.
The models are loaded sequentially to limit GPU memory use. For each context, a hook at the
final RMSNorm layer captures the residual vector before normalization and the corresponding
normalized vector. With right-padded batches, the attention-mask length identifies the last
real context token for each example.

The pre-RMSNorm vectors are the source and target representations for stitching: 3,072 dimensions
for Llama and 2,048 for Qwen. The post-RMSNorm vectors support normalization checks. Saved split
artifacts contain example manifests, per-model tensors, and a combined `paired.pt` payload with
ordered IDs, labels, representations, and provenance. The keys `h_llama` and `h_qwen` refer to
pre-RMSNorm vectors; the `_post` suffix identifies normalized representations.

This section describes the selected cache's schema, sample counts, tensor dimensions, and
precision, and connects each row to its manifest example. The final normalization and LM-head
parameters define the output computation used for native baselines and later prediction-based
objectives. Their reconstruction accuracy is evaluated in Phase 4.


In [5]:
import torch

HIDDEN_SIZES = {"llama": 3072, "qwen": 2048}


def load_paired_artifacts(directory, split, parent_rows, parent_meta):
    if split not in ("train", "validation"):
        raise ValueError("Artifact analysis is restricted to train and validation.")
    if directory is None:
        raise ValueError("Set CACHE_RUN_DIR to the saved extraction run printed by activation.ipynb.")
    folder = Path(directory) / split
    cache = torch.load(folder / "paired.pt", map_location="cpu", weights_only=True)
    meta = cache["provenance"]
    for key in ("dataset", "dataset_revision", "models", "revisions", "max_tokens"):
        if meta.get(key) != parent_meta[key]:
            raise ValueError(f"{split} cache differs from training provenance: {key}")
    if meta.get("split") != split:
        raise ValueError(f"Cache split mismatch: {split}")
    if cache.get("source_model") != parent_meta["models"]["llama"] or cache.get("target_model") != parent_meta["models"]["qwen"]:
        raise ValueError("Paired-cache model IDs do not match the selected dataset")
    rows = read_manifest(folder / "examples.jsonl", meta["manifest_sha256"])
    validate_manifest(rows, split, meta["max_tokens"])
    if len(rows) != meta["n_examples"] or cache["ids"] != [row["id"] for row in rows]:
        raise ValueError(f"Cache row count/ID order mismatch: {split}")
    if split == "train":
        parent_by_id = {row["id"]: row for row in parent_rows}
        if any(parent_by_id.get(row["id"]) != row for row in rows):
            raise ValueError("Cached training rows are not an exact subset of the selected training manifest")
    for name in ("llama", "qwen"):
        label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
        expected = torch.tensor([row[f"{name}_target_id"] for row in rows], dtype=torch.int64)
        labels = cache[label_key]
        if not isinstance(labels, torch.Tensor) or labels.dtype != torch.int64 or not torch.equal(labels, expected):
            raise ValueError(f"Cached {name} labels do not match manifest order/content")
        for suffix in ("", "_post"):
            key = f"h_{name}{suffix}"
            tensor = cache[key]
            if not isinstance(tensor, torch.Tensor) or tensor.shape != (len(rows), HIDDEN_SIZES[name]):
                raise ValueError(f"Invalid tensor shape: {split}/{key}")
            if tensor.device.type != "cpu" or not tensor.is_floating_point() or not torch.isfinite(tensor).all():
                raise ValueError(f"Invalid device/dtype/nonfinite activation: {split}/{key}")
    return rows, cache


def check_split_overlap(manifests):
    results = []
    for (left, left_rows), (right, right_rows) in combinations(manifests.items(), 2):
        ids = {row["id"] for row in left_rows} & {row["id"] for row in right_rows}
        contexts = {normalized_context(row) for row in left_rows} & {normalized_context(row) for row in right_rows}
        results.append([left, len(left_rows), right, len(right_rows), len(ids), len(contexts)])
        if ids or contexts:
            raise ValueError(f"Split overlap between {left} and {right}: {len(ids)} IDs, {len(contexts)} normalized contexts")
    return results


if not ANALYSIS_SPLITS or len(set(ANALYSIS_SPLITS)) != len(ANALYSIS_SPLITS) or any(s not in ("train", "validation") for s in ANALYSIS_SPLITS):
    raise ValueError("Select unique analysis splits from train and validation only.")
loaded = {split: load_paired_artifacts(CACHE_RUN_DIR, split, training_rows, training_meta)
          for split in ANALYSIS_SPLITS}
cache_manifests = {split: value[0] for split, value in loaded.items()}
paired_caches = {split: value[1] for split, value in loaded.items()}
overlap_rows = check_split_overlap({"Full training manifest": training_rows,
                                   **{f"Selected {s} manifest": rows for s, rows in cache_manifests.items() if s != "train"}})
cache_summary = []
for split, cache in paired_caches.items():
    for name in ("llama", "qwen"):
        for suffix, stage in (("", "pre-RMSNorm"), ("_post", "post-RMSNorm")):
            tensor = cache[f"h_{name}{suffix}"]
            cache_summary.append([split, name, stage, tensor.shape[0], tensor.shape[1], str(tensor.dtype),
                                  tensor.numel()*tensor.element_size()/1024**2,
                                  tensor.float().norm(dim=-1).mean().item()])
show_table("Cached representations", ["Split", "Model", "Stage", "Rows", "Width", "Dtype", "MiB", "Mean L2 norm"], cache_summary)
show_table("Cache sample coverage", ["Split", "Cached examples", "Available training examples"],
           [[split, len(cache_manifests[split]), len(training_rows) if split == "train" else "—"]
            for split in ANALYSIS_SPLITS])
show_table("Selected-manifest overlap", ["Manifest A", "Rows A", "Manifest B", "Rows B", "Shared IDs", "Shared contexts"], overlap_rows)
show_table("Cache provenance", ["Split", "Directory", "Manifest SHA256"],
           [[split, Path(CACHE_RUN_DIR)/split, paired_caches[split]["provenance"]["manifest_sha256"]]
            for split in ANALYSIS_SPLITS])
display(Markdown("Overlap comparisons cover only the manifests and sample counts shown above; "
                 "they do not establish source-book separation. The `_post` tensors are normalized "
                 "representations, not normalization parameters or output-head weights."))


### Cached representations

| Split | Model | Stage | Rows | Width | Dtype | MiB | Mean L2 norm |
| --- | --- | --- | --- | --- | --- | --- | --- |
| train | llama | pre-RMSNorm | 10000 | 3072 | torch.bfloat16 | 58.594 | 63.066 |
| train | llama | post-RMSNorm | 10000 | 3072 | torch.bfloat16 | 58.594 | 87.851 |
| train | qwen | pre-RMSNorm | 10000 | 2048 | torch.bfloat16 | 39.062 | 553.300 |
| train | qwen | post-RMSNorm | 10000 | 2048 | torch.bfloat16 | 39.062 | 171.531 |
| validation | llama | pre-RMSNorm | 4869 | 3072 | torch.bfloat16 | 28.529 | 66.666 |
| validation | llama | post-RMSNorm | 4869 | 3072 | torch.bfloat16 | 28.529 | 87.369 |
| validation | qwen | pre-RMSNorm | 4869 | 2048 | torch.bfloat16 | 19.020 | 585.719 |
| validation | qwen | post-RMSNorm | 4869 | 2048 | torch.bfloat16 | 19.020 | 165.830 |

### Cache sample coverage

| Split | Cached examples | Available training examples |
| --- | --- | --- |
| train | 10000 | 10000 |
| validation | 4869 | — |

### Selected-manifest overlap

| Manifest A | Rows A | Manifest B | Rows B | Shared IDs | Shared contexts |
| --- | --- | --- | --- | --- | --- |
| Full training manifest | 10000 | Selected validation manifest | 4869 | 0 | 0 |

### Cache provenance

| Split | Directory | Manifest SHA256 |
| --- | --- | --- |
| train | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train | 604387bf620dfe540501006eb7977fa73c99a31454f9e6093c0cc497747d3044 |
| validation | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation | d0d9208aab32b2b16c40368c538b654a00589e9ea834b1b7e5cb96824e3f5753 |

Overlap comparisons cover only the manifests and sample counts shown above; they do not establish source-book separation. The `_post` tensors are normalized representations, not normalization parameters or output-head weights.

## **Phase 3 — Native and Constant-Activation Baselines**

The baseline comparisons establish reference performance before fitting an affine stitch.
Native Llama and native Qwen use their own cached representations and corresponding normalization
and output heads. The constant-activation baseline predicts Qwen's mean training representation
for every context, then applies Qwen's normalization and output head. Only training examples
contribute to the mean.

Evaluation reports next-token cross-entropy and top-1 accuracy, along with accuracy and sample
counts for targets represented by a single token. Training and validation results are presented
separately. Llama's scores use its own tokenizer and are interpreted as a source-model reference;
agreement and KL comparisons with Qwen are computed within Qwen's vocabulary. Every metric is
associated with the selected manifest and its evaluation denominator.


In [6]:
import gc
import hashlib
import json
import math
import os
from importlib.metadata import version
from pathlib import Path
from tempfile import NamedTemporaryFile

import torch
import torch.nn.functional as F
from torch import nn
from tqdm.auto import tqdm

BASELINE_BATCH_SIZE = 32
BASELINE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def baseline_file_hash(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def baseline_object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def checked_artifact(path):
    path = Path(path)
    sidecar = path.with_suffix(".sha256.json")
    if not path.is_file() or not sidecar.is_file():
        raise FileNotFoundError(f"Expected exported artifact and checksum: {path}")
    if json.loads(sidecar.read_text())["sha256"] != baseline_file_hash(path):
        raise ValueError(f"Artifact checksum mismatch: {path}")
    return torch.load(path, map_location="cpu", weights_only=True)


def verify_exported_head(root, name, expected_model, expected_revision, run_hash, width):
    path = Path(root) / f"{name}_frozen_head.pt"
    state = checked_artifact(path)
    identity = state["provenance"]
    if identity["model"] != expected_model or identity["revision"] != expected_revision or identity["run_hash"] != run_hash:
        raise ValueError(f"Frozen {name} head does not belong to the selected extraction run")
    report = json.loads((Path(root) / f"{name}_head_validation.json").read_text())
    observations = report.get("observations", [])
    if report.get("provenance") != identity or report.get("passed") is not True or not observations:
        raise ValueError(f"Missing successful native-logit reconstruction evidence for {name}")
    if not all(o.get("allclose") is True and o.get("top1_match") is True for o in observations):
        raise ValueError(f"Head reconstruction checks failed for {name}")
    if state["hidden_size"] != width or tuple(state["norm_weight"].shape) != (width,):
        raise ValueError(f"Normalization dimensions differ from {name} activations")
    if tuple(state["lm_head_weight"].shape) != (state["vocab_size"], width):
        raise ValueError(f"Invalid {name} output-head dimensions")
    if not math.isfinite(state["norm_eps"]) or state["norm_eps"] <= 0:
        raise ValueError(f"Invalid {name} normalization epsilon")
    return state, report


class FrozenCausalHead(nn.Module):
    def __init__(self, state):
        super().__init__()
        self.eps = state["norm_eps"]
        self.register_buffer("norm_weight", state["norm_weight"])
        self.register_buffer("lm_head_weight", state["lm_head_weight"])

    def forward(self, hidden):
        # Preserve exported precision; accumulate variance and metric losses in fp32.
        hidden = hidden.to(self.norm_weight.dtype)
        normalized = hidden.float() * torch.rsqrt(hidden.float().pow(2).mean(-1, keepdim=True) + self.eps)
        normalized = normalized.to(hidden.dtype) * self.norm_weight
        return F.linear(normalized, self.lm_head_weight)


def new_metric_totals():
    return dict(n=0, ce_sum=0.0, correct=0, single_n=0, single_correct=0,
                agreement=0, kl_sum=0.0)


def accumulate_metrics(totals, logits, labels, single_mask):
    predictions = logits.argmax(-1)
    correct = predictions == labels
    totals["n"] += labels.numel()
    totals["ce_sum"] += F.cross_entropy(logits.float(), labels, reduction="sum").item()
    totals["correct"] += correct.sum().item()
    totals["single_n"] += single_mask.sum().item()
    totals["single_correct"] += correct[single_mask].sum().item()
    return predictions


def finalize_metrics(totals, qwen_reference=False):
    if not totals["n"]:
        raise ValueError("Cannot evaluate an empty split")
    return dict(n_examples=totals["n"], next_token_ce=totals["ce_sum"] / totals["n"],
                next_token_accuracy=totals["correct"] / totals["n"],
                single_token_count=totals["single_n"],
                single_token_accuracy=totals["single_correct"] / totals["single_n"] if totals["single_n"] else None,
                agreement_with_native_qwen=totals["agreement"] / totals["n"] if qwen_reference else None,
                kl_native_qwen_to_prediction=totals["kl_sum"] / totals["n"] if qwen_reference else None)


@torch.no_grad()
def evaluate_saved_representations(name, head, cache, rows, qwen_training_mean=None, batch_size=32):
    n = len(rows)
    device = head.norm_weight.device
    native_totals, constant_totals = new_metric_totals(), new_metric_totals()
    label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
    labels = cache[label_key]
    if labels.shape != (n,) or labels.dtype != torch.int64 or (labels < 0).any() or (labels >= head.lm_head_weight.shape[0]).any():
        raise ValueError(f"Invalid {name} labels for the exported vocabulary")
    constant_logits = None
    if name == "qwen":
        if qwen_training_mean is None:
            raise ValueError("The constant baseline requires the Qwen training mean")
        constant_logits = head(qwen_training_mean.to(device)).float()
        constant_logp = F.log_softmax(constant_logits, -1)
    for start in tqdm(range(0, n, batch_size), desc=f"{name} baselines", leave=False):
        end = min(start + batch_size, n)
        hidden = cache[f"h_{name}"][start:end].to(device)
        batch_labels = labels[start:end].to(device)
        single = torch.tensor([row[f"{name}_target_length"] == 1 for row in rows[start:end]], device=device)
        native_logits = head(hidden).float()
        native_preds = accumulate_metrics(native_totals, native_logits, batch_labels, single)
        if not torch.isfinite(native_logits).all():
            raise ValueError(f"Nonfinite native {name} logits")
        if name == "qwen":
            native_totals["agreement"] += end - start
            predicted = accumulate_metrics(constant_totals, constant_logits.expand(end-start, -1), batch_labels, single)
            constant_totals["agreement"] += (predicted == native_preds).sum().item()
            native_logp = F.log_softmax(native_logits, -1)
            constant_totals["kl_sum"] += (native_logp.exp() * (native_logp - constant_logp)).sum().item()
    result = {f"native_{name}": finalize_metrics(native_totals, name == "qwen")}
    if name == "qwen":
        result["constant_qwen_train_mean"] = finalize_metrics(constant_totals, True)
    return result


def save_baseline_results(root, report, mean_payload):
    directory = Path(root) / "baselines"
    directory.mkdir(parents=True, exist_ok=True)
    mean_path = directory / "qwen_training_mean.pt"
    with NamedTemporaryFile(dir=directory, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(mean_payload, temporary)
        os.replace(temporary, mean_path)
    finally:
        temporary.unlink(missing_ok=True)
    report["training_mean_sha256"] = baseline_file_hash(mean_path)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=directory, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        temporary.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
        os.replace(temporary, directory / "metrics.json")
    finally:
        temporary.unlink(missing_ok=True)
    return directory


def run_phase3_baselines(root, caches, manifests, parent_meta, widths, batch_size=32, device="cpu"):
    root = Path(root)
    if batch_size <= 0 or set(caches) != {"train", "validation"} or set(manifests) != set(caches):
        raise ValueError("Provide matching train and validation caches/manifests and a positive batch size")
    run_metadata = json.loads((root / "run.json").read_text())
    run_hash = baseline_object_hash(run_metadata["identity"])
    for split, cache in caches.items():
        checksum = json.loads((root / split / "paired.sha256.json").read_text())["sha256"]
        if checksum != baseline_file_hash(root / split / "paired.pt"):
            raise ValueError(f"Paired-cache checksum mismatch: {split}")
        if baseline_object_hash(cache["provenance"]) != run_metadata["identity"]["split_fingerprints"][split]:
            raise ValueError(f"Selected {split} cache does not match run.json")
        if cache["ids"] != [row["id"] for row in manifests[split]]:
            raise ValueError(f"Manifest/cache row order differs: {split}")
        for name in ("llama", "qwen"):
            h = cache[f"h_{name}"]
            label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
            expected = torch.tensor([row[f"{name}_target_id"] for row in manifests[split]])
            if h.shape != (len(manifests[split]), widths[name]) or not h.is_floating_point() or not torch.isfinite(h).all():
                raise ValueError(f"Invalid {split}/{name} activation matrix")
            if not torch.equal(cache[label_key], expected):
                raise ValueError(f"Invalid {split}/{name} label alignment")
    train_h = caches["train"]["h_qwen"]
    if not len(train_h):
        raise ValueError("Empty training cache")
    total = torch.zeros(widths["qwen"], dtype=torch.float64)
    for start in range(0, len(train_h), 256):
        total += train_h[start:start+256].double().sum(0)
    mean = (total / len(train_h)).float().unsqueeze(0)
    mean_payload = dict(mean=mean, n_train=len(train_h),
                        training_manifest_sha256=caches["train"]["provenance"]["manifest_sha256"],
                        training_ids_sha256=baseline_object_hash(caches["train"]["ids"]))
    report = dict(task="first continuation token", temperature=1.0, device=str(device), batch_size=batch_size,
                  runtime={p: version(p) for p in ("torch", "transformers", "huggingface_hub")},
                  source_artifact_dir=str(root), training_mean={k:v for k,v in mean_payload.items() if k!="mean"},
                  heads={}, results=[])
    for name in ("qwen", "llama"):
        head = state = None
        try:
            state, evidence = verify_exported_head(root, name, parent_meta["models"][name],
                parent_meta["revisions"][name], run_hash, widths[name])
            report["heads"][name] = dict(checksum=json.loads((root/f"{name}_frozen_head.sha256.json").read_text())["sha256"],
                                         provenance=state["provenance"], validation=evidence)
            head = FrozenCausalHead(state).eval().requires_grad_(False).to(device)
            del state
            state = None
            for split in ("train", "validation"):
                metrics = evaluate_saved_representations(name, head, caches[split], manifests[split],
                                                        mean if name == "qwen" else None, batch_size)
                for baseline, values in metrics.items():
                    report["results"].append(dict(baseline=baseline, split=split, tokenizer=parent_meta["models"][name],
                        manifest_sha256=caches[split]["provenance"]["manifest_sha256"], **values))
        finally:
            del head, state
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    baseline_order = {"native_qwen": 0, "native_llama": 1, "constant_qwen_train_mean": 2}
    report["results"].sort(key=lambda row: (("train", "validation").index(row["split"]), baseline_order[row["baseline"]]))
    output = save_baseline_results(root, report, mean_payload)
    return report, output


required_names = ("CACHE_RUN_DIR", "paired_caches", "cache_manifests", "training_meta", "HIDDEN_SIZES", "show_table")
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run the configuration and Phase 1/2 cells first. Missing: {missing}")
baseline_report, baseline_dir = run_phase3_baselines(
    CACHE_RUN_DIR, paired_caches, cache_manifests, training_meta, HIDDEN_SIZES,
    BASELINE_BATCH_SIZE, BASELINE_DEVICE,
)
show_table("Output-head reconstruction checks", ["Model", "Examples checked", "Maximum absolute logit error", "Top-1 matches"],
    [[name, len(item["validation"]["observations"]),
      max(o["max_abs_error"] for o in item["validation"]["observations"]),
      sum(o["top1_match"] for o in item["validation"]["observations"])]
     for name,item in baseline_report["heads"].items()])
show_table("Native and constant-activation baselines", ["Split", "Baseline", "N", "Next-token CE", "Top-1 accuracy", "Single-token N", "Single-token accuracy"],
    [[r["split"], r["baseline"], r["n_examples"], r["next_token_ce"], r["next_token_accuracy"],
      r["single_token_count"], r["single_token_accuracy"] if r["single_token_accuracy"] is not None else "—"]
     for r in baseline_report["results"]])
show_table(
    "Qwen-vocabulary comparison",
    ["Split", "Baseline", "Agreement with native Qwen",
     "KL(native Qwen → predicted)"],
    [
        [r["split"], r["baseline"],
         r["agreement_with_native_qwen"],
         r["kl_native_qwen_to_prediction"]]
        for r in baseline_report["results"]
        if r["agreement_with_native_qwen"] is not None
    ],
)
print(f"Training mean fitted on {baseline_report['training_mean']['n_train']:,} examples.")
print("Saved metrics and training mean to:", baseline_dir)


qwen baselines:   0%|          | 0/313 [00:00<?, ?it/s]

qwen baselines:   0%|          | 0/153 [00:00<?, ?it/s]

llama baselines:   0%|          | 0/313 [00:00<?, ?it/s]

llama baselines:   0%|          | 0/153 [00:00<?, ?it/s]

### Output-head reconstruction checks

| Model | Examples checked | Maximum absolute logit error | Top-1 matches |
| --- | --- | --- | --- |
| qwen | 3 | 0.062 | 3 |
| llama | 3 | 0.062 | 3 |

### Native and constant-activation baselines

| Split | Baseline | N | Next-token CE | Top-1 accuracy | Single-token N | Single-token accuracy |
| --- | --- | --- | --- | --- | --- | --- |
| train | native_qwen | 10000 | 2.180 | 0.484 | 7728 | 0.552 |
| train | native_llama | 10000 | 2.386 | 0.468 | 7740 | 0.529 |
| train | constant_qwen_train_mean | 10000 | 7.608 | 0.000 | 7728 | 0.000 |
| validation | native_qwen | 4869 | 1.699 | 0.600 | 2788 | 0.609 |
| validation | native_llama | 4869 | 1.453 | 0.653 | 2804 | 0.655 |
| validation | constant_qwen_train_mean | 4869 | 9.938 | 0.000 | 2788 | 0.000 |

### Qwen-vocabulary comparison

| Split | Baseline | Agreement with native Qwen | KL(native Qwen → predicted) |
| --- | --- | --- | --- |
| train | native_qwen | 1.000 | 0.000 |
| train | constant_qwen_train_mean | 0.033 | 4.363 |
| validation | native_qwen | 1.000 | 0.000 |
| validation | constant_qwen_train_mean | 0.048 | 5.691 |

Training mean fitted on 10,000 examples.
Saved metrics and training mean to: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/baselines


## **Phase 4 — Pipeline Validation**

Validation checks the correspondence between text, labels, cached representations, and reported
scores. Data checks cover source IDs, split overlap, token boundaries, label ranges, manifest
checksums, and paired row order. Tensor checks cover dimensions, precision, finite values, and
equality after saving and reloading.

Model checks compare batched extraction with single-example extraction and verify that applying
the saved normalization and output head to a pre-RMSNorm representation reproduces the original
model's logits within a stated numerical tolerance. Metric checks independently recompute a small
batch and verify that the constant representation uses training data only. The analysis records
the example IDs, observed differences, tolerances, and artifact references used in each comparison.


In [7]:
import gc
import json
import math
import os
import random
from pathlib import Path
from tempfile import NamedTemporaryFile

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# Prerequisite check: ensure Config and Phases 1-3 have run in this session
required_p4 = (
    "CACHE_RUN_DIR", "ANALYSIS_SPLITS", "paired_caches", "cache_manifests",
    "training_meta", "HIDDEN_SIZES", "show_table", "normalized_context",
    "BASELINE_DEVICE", "checked_artifact", "baseline_file_hash",
    "baseline_object_hash", "verify_exported_head", "FrozenCausalHead",
    "evaluate_saved_representations",
)
missing_p4 = [name for name in required_p4 if name not in globals()]
if missing_p4:
    raise RuntimeError(f"Run Config and Phases 1-3 first. Missing: {missing_p4}")

# Set to True on an L4 GPU runtime (with HF_TOKEN enabled) to run live model parity checks
RUN_LIVE_MODEL_CHECKS = True

TOKENIZER_SAMPLE_N = 100     # Examples per split to re-tokenize from scratch
METRIC_CHECK_N = 32          # Validation examples for independent metric recomputation
LIVE_CHECK_N = 4             # Varying-length validation examples for batched vs. single checks

NORM_ATOL, NORM_RTOL = 1e-2, 1e-2          # bfloat16 pre->post RMSNorm tolerance
ACT_MAX_REL_L2 = 0.03                     # Per-example relative vector error (3%)
ACT_MAX_NORM_REL_ERR = 0.02               # Per-example relative norm error (2%)
ACT_MIN_COSINE = 0.999                     # Batched vs. single-example activation cosine floor
HEAD_ATOL, HEAD_RTOL = 1e-1, 1e-2          # bfloat16 single-example head reconstruction tolerance
CACHED_LOGIT_MIN_COS = 0.9995              # End-to-end cached->native logit cosine floor
METRIC_ATOL = 1e-5                         # float32 metric recomputation tolerance

val_checks = []
val_details = {
    "artifact_dir": str(CACHE_RUN_DIR),
    "analysis_splits": list(ANALYSIS_SPLITS),
    "run_live_model_checks": bool(RUN_LIVE_MODEL_CHECKS),
    "checks": {},
}



def activation_parity(candidate, reference):
    """Check both direction and scale, using the single-example vector as reference."""
    if candidate.shape != reference.shape or candidate.ndim != 2 or not len(candidate):
        raise ValueError("Activation comparisons require matching nonempty [N, D] tensors")
    candidate, reference = candidate.float(), reference.float()
    if not torch.isfinite(candidate).all() or not torch.isfinite(reference).all():
        return dict(passed=False, min_cosine=0.0, max_rel_l2=1e30, max_norm_rel_err=1e30)
    eps = 1e-12
    reference_norm = reference.norm(dim=-1)
    candidate_norm = candidate.norm(dim=-1)
    denominator = reference_norm.clamp_min(eps)
    relative_l2 = (candidate-reference).norm(dim=-1) / denominator
    norm_error = (candidate_norm-reference_norm).abs() / denominator
    cosine = F.cosine_similarity(candidate, reference, dim=-1)
    cosine = torch.where((reference_norm <= eps) & (candidate_norm <= eps), torch.ones_like(cosine), cosine)
    values = dict(min_cosine=cosine.min().item(), max_rel_l2=relative_l2.max().item(),
                  max_norm_rel_err=norm_error.max().item())
    values["passed"] = (values["min_cosine"] >= ACT_MIN_COSINE
                        and values["max_rel_l2"] <= ACT_MAX_REL_L2
                        and values["max_norm_rel_err"] <= ACT_MAX_NORM_REL_ERR)
    return values


def compare_metric_result(actual, expected):
    """Require exact counts, the correct None policy, and finite numerical metrics."""
    differences, issues = {}, []
    for key, reference in expected.items():
        if key not in actual:
            issues.append(f"{key}: missing")
            continue
        observed = actual[key]
        if key in ("n_examples", "single_token_count"):
            if type(observed) is not int or observed != reference:
                issues.append(f"{key}: expected {reference}, got {observed!r}")
        elif reference is None:
            if observed is not None:
                issues.append(f"{key}: expected None, got {observed!r}")
        elif type(observed) not in (int, float) or not math.isfinite(observed):
            issues.append(f"{key}: expected a finite number, got {observed!r}")
        else:
            differences[key] = abs(observed-reference)
    return differences, issues


def record_check(name, passed, observed, tolerance, evidence, ids=None):
    val_checks.append([name, "PASS" if passed else "FAIL", observed, tolerance, evidence])
    entry = {
        "passed": bool(passed),
        "observed": observed,
        "tolerance": tolerance,
        "evidence": evidence,
    }
    if ids is not None:
        entry["example_ids"] = ids
    val_details["checks"][name] = entry


# -------------------------------------------------------------------------
# 1. Data, manifest, split-overlap, tensor, and disk-reload integrity checks
# -------------------------------------------------------------------------
run_meta = json.loads((CACHE_RUN_DIR / "run.json").read_text())
run_hash = baseline_object_hash(run_meta["identity"])
expected_dtype = run_meta["settings"]["compute_dtype"]

head_states = {}
for name in ("qwen", "llama"):
    state, _ = verify_exported_head(
        CACHE_RUN_DIR, name,
        training_meta["models"][name],
        training_meta["revisions"][name],
        run_hash, HIDDEN_SIZES[name],
    )
    head_states[name] = state

# Explicit split-overlap check (train vs. validation)
train_ids_set = {r["id"] for r in cache_manifests["train"]}
val_ids_set = {r["id"] for r in cache_manifests["validation"]}
train_ctx_set = {normalized_context(r) for r in cache_manifests["train"]}
val_ctx_set = {normalized_context(r) for r in cache_manifests["validation"]}
shared_ids = len(train_ids_set & val_ids_set)
shared_ctx = len(train_ctx_set & val_ctx_set)
record_check(
    "Split separation (train vs. validation)",
    shared_ids == 0 and shared_ctx == 0,
    f"shared_ids={shared_ids}, shared_contexts={shared_ctx}",
    "0 shared IDs & 0 shared contexts",
    f"train={len(train_ids_set)}, val={len(val_ids_set)}",
)

for split in ANALYSIS_SPLITS:
    rows = cache_manifests[split]
    cache = paired_caches[split]
    split_dir = CACHE_RUN_DIR / split

    # Manifest checksum, source IDs, paired row order, label ranges
    manifest_sha_ok = baseline_file_hash(split_dir / "examples.jsonl") == cache["provenance"]["manifest_sha256"]
    ids_unique = len(cache["ids"]) == len(set(cache["ids"]))
    row_order_ok = cache["ids"] == [r["id"] for r in rows]
    id_prefix_ok = all(r["id"].startswith(f"{split}/") for r in rows)

    q_labels = cache["target_token_ids"]
    l_labels = cache["llama_target_token_ids"]
    q_range_ok = bool((q_labels >= 0).all() and (q_labels < head_states["qwen"]["vocab_size"]).all())
    l_range_ok = bool((l_labels >= 0).all() and (l_labels < head_states["llama"]["vocab_size"]).all())
    labels_match_manifest = (
        torch.equal(q_labels, torch.tensor([r["qwen_target_id"] for r in rows], dtype=torch.int64))
        and torch.equal(l_labels, torch.tensor([r["llama_target_id"] for r in rows], dtype=torch.int64))
    )

    data_ok = all([manifest_sha_ok, ids_unique, row_order_ok, id_prefix_ok, q_range_ok, l_range_ok, labels_match_manifest])
    record_check(
        f"Data & label alignment ({split})",
        data_ok,
        f"sha={manifest_sha_ok}, order={row_order_ok}, labels={labels_match_manifest}, ranges={q_range_ok and l_range_ok}",
        "all True",
        f"n={len(rows)}, manifest={split_dir / 'examples.jsonl'}",
    )

    # Tensor shapes, precision, finiteness, and fresh disk reload parity
    reloaded = checked_artifact(split_dir / "paired.pt")
    tensor_ok = True
    reload_exact = reloaded["ids"] == cache["ids"]
    for name, width in HIDDEN_SIZES.items():
        for suffix in ("", "_post"):
            k = f"h_{name}{suffix}"
            t = cache[k]
            if (
                t.shape != (len(rows), width)
                or str(t.dtype) != expected_dtype
                or not torch.isfinite(t).all()
            ):
                tensor_ok = False
            if not torch.equal(reloaded[k], t):
                reload_exact = False
    reload_exact = reload_exact and torch.equal(reloaded["target_token_ids"], q_labels) and torch.equal(reloaded["llama_target_token_ids"], l_labels)
    del reloaded

    record_check(
        f"Tensor & disk-reload integrity ({split})",
        tensor_ok and reload_exact,
        f"shapes_dtype_finite={tensor_ok}, disk_reload_exact={reload_exact}",
        f"{expected_dtype}, finite, bitwise torch.equal",
        str(split_dir / "paired.pt"),
    )


# -------------------------------------------------------------------------
# 2. Train-only constant mean provenance & recomputation check
# -------------------------------------------------------------------------
mean_path = CACHE_RUN_DIR / "baselines" / "qwen_training_mean.pt"
metrics_path = CACHE_RUN_DIR / "baselines" / "metrics.json"
metrics_json = json.loads(metrics_path.read_text())
mean_hash_ok = baseline_file_hash(mean_path) == metrics_json["training_mean_sha256"]
saved_mean = torch.load(mean_path, map_location="cpu", weights_only=True)

recomputed_train_mean = paired_caches["train"]["h_qwen"].double().mean(dim=0, keepdim=True).float()
val_mean = paired_caches["validation"]["h_qwen"].double().mean(dim=0, keepdim=True).float()

mean_diff = (saved_mean["mean"] - recomputed_train_mean).abs().max().item()
val_diff = (saved_mean["mean"] - val_mean).abs().max().item()
mean_ok = (
    mean_hash_ok
    and saved_mean["n_train"] == len(paired_caches["train"]["ids"])
    and saved_mean["training_manifest_sha256"] == paired_caches["train"]["provenance"]["manifest_sha256"]
    and saved_mean["training_ids_sha256"] == baseline_object_hash(paired_caches["train"]["ids"])
    and mean_diff <= 1e-6
)
record_check(
    "Train-only constant mean provenance",
    mean_ok,
    f"train_diff={mean_diff:.2e}, val_diff={val_diff:.3f}",
    "<= 1e-6 vs train; validation difference is descriptive only",
    f"n_train={saved_mean['n_train']}, sha256_ok={mean_hash_ok}, file={mean_path.name}",
)


# -------------------------------------------------------------------------
# 3. Pre-RMSNorm -> Post-RMSNorm consistency using exported heads
# -------------------------------------------------------------------------
for name in ("qwen", "llama"):
    head_cpu = FrozenCausalHead(head_states[name]).eval()
    for split in ANALYSIS_SPLITS:
        pre = paired_caches[split][f"h_{name}"].to(head_cpu.norm_weight.dtype)
        post = paired_caches[split][f"h_{name}_post"]
        recomputed_post = (
            pre.float() * torch.rsqrt(pre.float().pow(2).mean(-1, keepdim=True) + head_cpu.eps)
        ).to(pre.dtype) * head_cpu.norm_weight
        max_err = (recomputed_post.float() - post.float()).abs().max().item()
        ok = torch.allclose(recomputed_post.float(), post.float(), atol=NORM_ATOL, rtol=NORM_RTOL)
        record_check(
            f"Pre->post RMSNorm ({name}, {split})",
            ok,
            f"max_abs_err={max_err:.5f}",
            f"atol={NORM_ATOL}, rtol={NORM_RTOL}",
            f"all {len(pre)} cached rows",
        )


# -------------------------------------------------------------------------
# 4. Independent tokenizer boundary & label re-verification
# -------------------------------------------------------------------------
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token is not None:
    from huggingface_hub import login
    login(token=hf_token)

rng = random.Random(42)
tokenizers_p4 = {}
try:
    for name, repo in training_meta["models"].items():
        tokenizers_p4[name] = AutoTokenizer.from_pretrained(repo, revision=training_meta["revisions"][name])

    for split in ANALYSIS_SPLITS:
        rows = cache_manifests[split]
        sample_idxs = sorted(rng.sample(range(len(rows)), min(TOKENIZER_SAMPLE_N, len(rows))))
        sample_ids = [rows[i]["id"] for i in sample_idxs]
        mismatches = 0
        for idx in sample_idxs:
            row = rows[idx]
            full_text = row["prefix"] + row["continuation"]
            for name, tok in tokenizers_p4.items():
                p_ids = tok.encode(row["prefix"])
                f_ids = tok.encode(full_text)
                if (
                    len(p_ids) != row[f"{name}_length"]
                    or len(p_ids) > training_meta["max_tokens"]
                    or f_ids[:len(p_ids)] != p_ids
                    or len(f_ids) <= len(p_ids)
                    or f_ids[len(p_ids)] != row[f"{name}_target_id"]
                    or (len(f_ids) - len(p_ids)) != row[f"{name}_target_length"]
                ):
                    mismatches += 1
        record_check(
            f"Live tokenizer boundary audit ({split})",
            mismatches == 0,
            f"mismatches={mismatches}/{len(sample_idxs) * 2}",
            "0 mismatches",
            f"n={len(sample_idxs)}, first_ids={sample_ids[:3]}",
            ids=sample_ids,
        )
except Exception as exc:
    record_check(
        "Live tokenizer boundary audit",
        False,
        f"ERROR: {type(exc).__name__}",
        "0 mismatches",
        str(exc),
    )


# -------------------------------------------------------------------------
# 5. Independent small-batch metric recomputation (Qwen & Llama validation)
# -------------------------------------------------------------------------
class MetricAuditHead(torch.nn.Module):
    """Capture the exact pipeline logits so the audit isolates metric arithmetic."""
    def __init__(self, head):
        super().__init__()
        self.head = head
        self.calls = []

    @property
    def norm_weight(self):
        return self.head.norm_weight

    @property
    def lm_head_weight(self):
        return self.head.lm_head_weight

    def forward(self, hidden):
        logits = self.head(hidden)
        self.calls.append(logits.detach().to(device="cpu", dtype=torch.float32).clone())
        return logits


val_rows_sub = cache_manifests["validation"][:METRIC_CHECK_N]
metric_check_ids = [r["id"] for r in val_rows_sub]

for name in ("qwen", "llama"):
    label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
    val_cache_sub = {
        "ids": paired_caches["validation"]["ids"][:METRIC_CHECK_N],
        f"h_{name}": paired_caches["validation"][f"h_{name}"][:METRIC_CHECK_N],
        label_key: paired_caches["validation"][label_key][:METRIC_CHECK_N],
    }
    head_dev = MetricAuditHead(FrozenCausalHead(head_states[name]).eval().to(BASELINE_DEVICE))
    pipeline_res = evaluate_saved_representations(
        name, head_dev, val_cache_sub, val_rows_sub,
        qwen_training_mean=saved_mean["mean"] if name == "qwen" else None,
        batch_size=8,
    )

    expected_batch_sizes = ([1] if name == "qwen" else []) + [
        min(8, len(val_rows_sub)-start) for start in range(0, len(val_rows_sub), 8)
    ]
    if [len(logits) for logits in head_dev.calls] != expected_batch_sizes:
        raise ValueError("Unexpected head-call layout during the independent metric audit")
    native_logits_ref = torch.cat(head_dev.calls[1:] if name == "qwen" else head_dev.calls, dim=0)
    if not torch.isfinite(native_logits_ref).all():
        raise ValueError("Nonfinite logits in the independent metric audit")

    ce_list, acc_list, single_acc_list = [], [], []
    const_ce_list, const_acc_list, const_single_acc_list, agree_list, kl_list = [], [], [], [], []
    with torch.no_grad():
        if name == "qwen":
            # Use the pipeline's recorded constant logits; no second low-precision head pass.
            const_logits_ref = head_dev.calls[0].double()
            const_logp_ref = F.log_softmax(const_logits_ref, dim=-1)
            const_pred = const_logits_ref.argmax(-1).item()

        for i, row in enumerate(val_rows_sub):
            y_i = val_cache_sub[label_key][i:i+1].cpu()
            # Independent per-example fp64 arithmetic on identical pipeline logits.
            # Live model checks below separately audit representation/head parity.
            logits_i = native_logits_ref[i:i+1].double()
            pred_i = logits_i.argmax(-1).item()

            ce_list.append(F.cross_entropy(logits_i, y_i).item())
            hit = int(pred_i == y_i.item())
            acc_list.append(hit)
            if row[f"{name}_target_length"] == 1:
                single_acc_list.append(hit)

            if name == "qwen":
                const_ce_list.append(F.cross_entropy(const_logits_ref, y_i).item())
                const_hit = int(const_pred == y_i.item())
                const_acc_list.append(const_hit)
                if row[f"{name}_target_length"] == 1:
                    const_single_acc_list.append(const_hit)
                agree_list.append(int(const_pred == pred_i))
                logp_i = F.log_softmax(logits_i, dim=-1)
                kl_list.append((logp_i.exp() * (logp_i - const_logp_ref)).sum().item())


    native_key = f"native_{name}"
    expected_native = {
        "n_examples": len(val_rows_sub),
        "next_token_ce": sum(ce_list) / len(ce_list),
        "next_token_accuracy": sum(acc_list) / len(acc_list),
        "single_token_count": len(single_acc_list),
        "single_token_accuracy": sum(single_acc_list) / len(single_acc_list) if single_acc_list else None,
        "agreement_with_native_qwen": 1.0 if name == "qwen" else None,
        "kl_native_qwen_to_prediction": 0.0 if name == "qwen" else None,
    }
    differences, issues = compare_metric_result(pipeline_res.get(native_key, {}), expected_native)
    diffs = {f"native_{key}": value for key, value in differences.items()}
    issues = [f"native: {issue}" for issue in issues]
    if name == "qwen":
        expected_constant = {
            "n_examples": len(val_rows_sub),
            "next_token_ce": sum(const_ce_list) / len(const_ce_list),
            "next_token_accuracy": sum(const_acc_list) / len(const_acc_list),
            "single_token_count": len(const_single_acc_list),
            "single_token_accuracy": sum(const_single_acc_list) / len(const_single_acc_list) if const_single_acc_list else None,
            "agreement_with_native_qwen": sum(agree_list) / len(agree_list),
            "kl_native_qwen_to_prediction": sum(kl_list) / len(kl_list),
        }
        differences, constant_issues = compare_metric_result(
            pipeline_res.get("constant_qwen_train_mean", {}), expected_constant
        )
        diffs.update({f"constant_{key}": value for key, value in differences.items()})
        issues.extend(f"constant: {issue}" for issue in constant_issues)

    max_metric_diff = max(diffs.values(), default=0.0)
    check_name = f"Independent metric recomputation ({name}, n={len(val_rows_sub)})"
    record_check(
        check_name,
        not issues and max_metric_diff <= METRIC_ATOL,
        f"max_diff={max_metric_diff:.2e}, field_issues={len(issues)}",
        f"exact counts/None policy; finite metrics; differences <= {METRIC_ATOL}",
        f"first_ids={metric_check_ids[:3]}; issues={issues}",
        ids=metric_check_ids,
    )
    val_details["checks"][check_name]["audit_method"] = "shared-logits-fp64-v1"
    val_details["checks"][check_name]["metric_differences"] = diffs
    val_details["checks"][check_name]["field_issues"] = issues
    del head_dev, native_logits_ref
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# -------------------------------------------------------------------------
# 6. Live GPU Model Checks: Batched vs. Single-Example & Native Logits
# -------------------------------------------------------------------------
if RUN_LIVE_MODEL_CHECKS:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable an L4 GPU runtime to run RUN_LIVE_MODEL_CHECKS=True.")
    if set(tokenizers_p4) != {"llama", "qwen"}:
        raise RuntimeError("Both tokenizers must load (verify HF_TOKEN) before running live GPU checks.")

    val_rows = cache_manifests["validation"]
    by_len = sorted(range(len(val_rows)), key=lambda i: val_rows[i]["qwen_length"])
    pick_positions = sorted({int(k * (len(by_len) - 1) / max(1, LIVE_CHECK_N - 1)) for k in range(LIVE_CHECK_N)})
    live_idxs = [by_len[p] for p in pick_positions]
    live_rows = [val_rows[i] for i in live_idxs]
    live_ids = [r["id"] for r in live_rows]

    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    attn_impl = run_meta["settings"]["attention"]

    for name, repo in training_meta["models"].items():
        tok = tokenizers_p4[name]
        tok.padding_side = "right"
        tok.pad_token = tok.eos_token
        head_gpu = FrozenCausalHead(head_states[name]).eval().to("cuda")
        model = AutoModelForCausalLM.from_pretrained(
            repo,
            revision=training_meta["revisions"][name],
            dtype=compute_dtype,
            attn_implementation=attn_impl,
        ).eval().requires_grad_(False).to("cuda")

        min_cos_batch_vs_single = 1.0
        min_cos_cache_vs_single = 1.0
        batch_activation_checks, cached_activation_checks = [], []
        max_head_err = 0.0
        head_top1_matches = 0
        head_allclose = True
        max_cached_logit_err = 0.0
        min_cached_logit_cos = 1.0
        cached_top1_matches = 0

        try:
            with torch.inference_mode():
                # Right-padded batch extraction matching activation_full.ipynb
                batch_inputs = tok([r["prefix"] for r in live_rows], padding=True, return_tensors="pt").to("cuda")
                batch_lengths = batch_inputs.attention_mask.sum(1)
                assert batch_lengths.tolist() == [r[f"{name}_length"] for r in live_rows]
                batch_idx = (torch.arange(len(live_rows), device="cuda"), batch_lengths - 1)
                batch_captured = {}

                def batch_pre_hook(module, args):
                    batch_captured["pre"] = args[0][batch_idx].detach().cpu()

                b_handle = model.model.norm.register_forward_pre_hook(batch_pre_hook)
                try:
                    _ = model.model(**batch_inputs, use_cache=False)
                finally:
                    b_handle.remove()

                for pos_in_batch, idx in enumerate(live_idxs):
                    row = val_rows[idx]
                    inputs = tok(row["prefix"], return_tensors="pt").to("cuda")
                    single_captured = {}

                    def single_pre_hook(module, args):
                        single_captured["pre"] = args[0][:, -1, :].detach().cpu()

                    s_handle = model.model.norm.register_forward_pre_hook(single_pre_hook)
                    try:
                        native_logits = model(**inputs, use_cache=False).logits[:, -1, :].detach().cpu().float()
                    finally:
                        s_handle.remove()

                    cached_pre = paired_caches["validation"][f"h_{name}"][idx:idx+1]
                    single_pre = single_captured["pre"]
                    padded_batch_pre = batch_captured["pre"][pos_in_batch:pos_in_batch+1]

                    # Activation parity: right-padded batch vs single, and cached vs single
                    batch_stats = activation_parity(padded_batch_pre, single_pre)
                    cache_stats = activation_parity(cached_pre, single_pre)
                    batch_activation_checks.append(dict(id=row["id"], **batch_stats))
                    cached_activation_checks.append(dict(id=row["id"], **cache_stats))
                    min_cos_batch_vs_single = min(min_cos_batch_vs_single, batch_stats["min_cosine"])
                    min_cos_cache_vs_single = min(min_cos_cache_vs_single, cache_stats["min_cosine"])

                    # Pure frozen-head logit reconstruction (single_pre -> head vs native_logits)
                    head_logits = head_gpu(single_pre.to("cuda")).detach().cpu().float()
                    max_head_err = max(max_head_err, (native_logits - head_logits).abs().max().item())
                    head_top1_matches += int(native_logits.argmax(-1).item() == head_logits.argmax(-1).item())
                    head_allclose = head_allclose and torch.allclose(
                        native_logits, head_logits, atol=HEAD_ATOL, rtol=HEAD_RTOL
                    )

                    # End-to-end cached activation -> frozen head vs native_logits
                    cached_logits = head_gpu(cached_pre.to("cuda")).detach().cpu().float()
                    max_cached_logit_err = max(max_cached_logit_err, (native_logits - cached_logits).abs().max().item())
                    min_cached_logit_cos = min(min_cached_logit_cos, F.cosine_similarity(native_logits, cached_logits).item())
                    cached_top1_matches += int(native_logits.argmax(-1).item() == cached_logits.argmax(-1).item())
        finally:
            del model, head_gpu
            gc.collect()
            torch.cuda.empty_cache()

        record_check(
            f"Batched vs. single activation ({name})",
            all(item["passed"] for item in batch_activation_checks + cached_activation_checks),
            (f"batch: cos={min_cos_batch_vs_single:.6f}, "
             f"rel_L2={max(item['max_rel_l2'] for item in batch_activation_checks):.4f}, "
             f"norm_rel={max(item['max_norm_rel_err'] for item in batch_activation_checks):.4f}; "
             f"cache: cos={min_cos_cache_vs_single:.6f}, "
             f"rel_L2={max(item['max_rel_l2'] for item in cached_activation_checks):.4f}, "
             f"norm_rel={max(item['max_norm_rel_err'] for item in cached_activation_checks):.4f}"),
            f"cos >= {ACT_MIN_COSINE}; rel_L2 <= {ACT_MAX_REL_L2}; norm_rel <= {ACT_MAX_NORM_REL_ERR}",
            f"ids={live_ids}",
            ids=live_ids,
        )
        val_details["checks"][f"Batched vs. single activation ({name})"]["per_example_errors"] = {
            "batch_vs_single": batch_activation_checks,
            "cache_vs_single": cached_activation_checks,
        }
        record_check(
            f"Frozen-head logit reconstruction ({name})",
            head_allclose and head_top1_matches == len(live_idxs),
            f"max_err={max_head_err:.4f}, top1={head_top1_matches}/{len(live_idxs)}",
            f"atol={HEAD_ATOL}, rtol={HEAD_RTOL}, 100% top1",
            f"ids={live_ids}, head={name}_frozen_head.pt",
            ids=live_ids,
        )
        record_check(
            f"End-to-end cached->native logits ({name})",
            min_cached_logit_cos >= CACHED_LOGIT_MIN_COS and cached_top1_matches == len(live_idxs),
            f"min_cos={min_cached_logit_cos:.6f}, max_err={max_cached_logit_err:.4f}, top1={cached_top1_matches}/{len(live_idxs)}",
            f"cos >= {CACHED_LOGIT_MIN_COS}, 100% top1",
            f"ids={live_ids}",
            ids=live_ids,
        )
else:
    record_check(
        "Live GPU checks (batched vs. single & native logits)",
        False,
        "NOT RUN (RUN_LIVE_MODEL_CHECKS=False)",
        f"act_cos >= {ACT_MIN_COSINE}; head atol={HEAD_ATOL}",
        "Set RUN_LIVE_MODEL_CHECKS=True on L4 GPU to complete",
    )


# -------------------------------------------------------------------------
# 7. Display & save validation report (atomic write)
# -------------------------------------------------------------------------
show_table(
    "Phase 4 — Pipeline Validation Report",
    ["Check", "Status", "Observed", "Tolerance", "Evidence"],
    val_checks,
)

report_dir = CACHE_RUN_DIR / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "phase4_validation_report.json"
with NamedTemporaryFile(mode="w", encoding="utf-8", dir=report_dir, delete=False) as tmp:
    tmp_path = Path(tmp.name)
    json.dump(val_details, tmp, indent=2, allow_nan=False)
    tmp.write("\n")
try:
    os.replace(tmp_path, report_path)
finally:
    tmp_path.unlink(missing_ok=True)
print("Saved validation report to:", report_path)

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

... (13 outputs left out)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

### Phase 4 — Pipeline Validation Report

| Check | Status | Observed | Tolerance | Evidence |
| --- | --- | --- | --- | --- |
| Split separation (train vs. validation) | PASS | shared_ids=0, shared_contexts=0 | 0 shared IDs & 0 shared contexts | train=10000, val=4869 |
| Data & label alignment (train) | PASS | sha=True, order=True, labels=True, ranges=True | all True | n=10000, manifest=/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train/examples.jsonl |
| Tensor & disk-reload integrity (train) | PASS | shapes_dtype_finite=True, disk_reload_exact=True | torch.bfloat16, finite, bitwise torch.equal | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train/paired.pt |
| Data & label alignment (validation) | PASS | sha=True, order=True, labels=True, ranges=True | all True | n=4869, manifest=/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation/examples.jsonl |
| Tensor & disk-reload integrity (validation) | PASS | shapes_dtype_finite=True, disk_reload_exact=True | torch.bfloat16, finite, bitwise torch.equal | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation/paired.pt |
| Train-only constant mean provenance | PASS | train_diff=0.00e+00, val_diff=36.242 | <= 1e-6 vs train; validation difference is descriptive only | n_train=10000, sha256_ok=True, file=qwen_training_mean.pt |
| Pre->post RMSNorm (qwen, train) | PASS | max_abs_err=0.03125 | atol=0.01, rtol=0.01 | all 10000 cached rows |
| Pre->post RMSNorm (qwen, validation) | PASS | max_abs_err=0.06250 | atol=0.01, rtol=0.01 | all 4869 cached rows |
| Pre->post RMSNorm (llama, train) | PASS | max_abs_err=0.03125 | atol=0.01, rtol=0.01 | all 10000 cached rows |
| Pre->post RMSNorm (llama, validation) | PASS | max_abs_err=0.06250 | atol=0.01, rtol=0.01 | all 4869 cached rows |
| Live tokenizer boundary audit (train) | PASS | mismatches=0/200 | 0 mismatches | n=100, first_ids=['train/1110/140636/140969', 'train/90/442729/443074', 'train/1409/39215/39503'] |
| Live tokenizer boundary audit (validation) | PASS | mismatches=0/200 | 0 mismatches | n=100, first_ids=['validation/4', 'validation/26', 'validation/58'] |
| Independent metric recomputation (qwen, n=32) | PASS | max_diff=2.19e-06, field_issues=0 | exact counts/None policy; finite metrics; differences <= 1e-05 | first_ids=['validation/0', 'validation/1', 'validation/2']; issues=[] |
| Independent metric recomputation (llama, n=32) | PASS | max_diff=3.79e-07, field_issues=0 | exact counts/None policy; finite metrics; differences <= 1e-05 | first_ids=['validation/0', 'validation/1', 'validation/2']; issues=[] |
| Batched vs. single activation (llama) | PASS | batch: cos=0.999876, rel_L2=0.0166, norm_rel=0.0049; cache: cos=0.999912, rel_L2=0.0143, norm_rel=0.0053 | cos >= 0.999; rel_L2 <= 0.03; norm_rel <= 0.02 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'] |
| Frozen-head logit reconstruction (llama) | PASS | max_err=0.0625, top1=4/4 | atol=0.1, rtol=0.01, 100% top1 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'], head=llama_frozen_head.pt |
| End-to-end cached->native logits (llama) | PASS | min_cos=0.999934, max_err=0.1562, top1=4/4 | cos >= 0.9995, 100% top1 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'] |
| Batched vs. single activation (qwen) | PASS | batch: cos=0.999915, rel_L2=0.0132, norm_rel=0.0036; cache: cos=0.999910, rel_L2=0.0135, norm_rel=0.0036 | cos >= 0.999; rel_L2 <= 0.03; norm_rel <= 0.02 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'] |
| Frozen-head logit reconstruction (qwen) | PASS | max_err=0.0625, top1=4/4 | atol=0.1, rtol=0.01, 100% top1 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'], head=qwen_frozen_head.pt |
| End-to-end cached->native logits (qwen) | PASS | min_cos=0.999765, max_err=0.2500, top1=4/4 | cos >= 0.9995, 100% top1 | ids=['validation/2952', 'validation/4866', 'validation/3464', 'validation/2265'] |

Saved validation report to: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/reports/phase4_validation_report.json


## **Phase 5 — Prep for MSE/CE/KL experiments**

The Milestone 2 interface connects paired activation caches to an affine transformation from
Llama's 3,072-dimensional representation to Qwen's 2,048-dimensional representation. A model-free
loader supplies aligned source activations, target activations, Qwen labels, and example IDs.
The affine scaffold specifies the mapping and initialization without training it in this notebook.

MSE compares predicted and native Qwen pre-RMSNorm activations. CE evaluates the predicted
representation through Qwen's frozen normalization and output head against the ground-truth
Qwen label. KL compares the resulting distribution with native Qwen's distribution, which can
be recomputed batchwise from cached Qwen activations through the same output components.

This section documents those interfaces and summarizes the dataset, cache schema, baseline
comparisons, and validation results used for the objective comparison. It also identifies the
exact artifacts and split definitions so the experiments share a consistent evaluation framework.


In [8]:
# Phase 5 — artifact handoff, data loaders, and untrained affine stitch.
# Run after Phases 1–4 in the same session. No backbone loading or optimizer steps.

"""Milestone 2 inputs and objectives. No backbone loading or training on import."""
import hashlib
import json
import math
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def load_handoff(path):
    """Verify the complete inventory in a fresh runtime before loading any weights."""
    path = Path(path)
    report = json.loads(path.read_text())
    if report.get('schema') != 'cross-family-milestone2-v2':
        raise ValueError('Unsupported handoff schema')
    for key, item in report['artifacts'].items():
        if file_hash(item['path']) != item['sha256']:
            raise ValueError(f'Handoff artifact changed: {key}')
    return report


def read_split(cache_path, manifest_path, split, fingerprint, metadata, widths, vocab_sizes):
    """Check alignment without tokenizers; retain repeated official evaluation contexts."""
    path = Path(cache_path)
    checksum = json.loads(path.with_suffix('.sha256.json').read_text())['sha256']
    if file_hash(path) != checksum:
        raise ValueError(f'Paired-cache checksum mismatch: {split}')
    cache = torch.load(path, map_location='cpu', weights_only=True)
    provenance = cache['provenance']
    if object_hash(provenance) != fingerprint or provenance.get('split') != split:
        raise ValueError(f'Wrong cache fingerprint/split: {split}')
    for key in ('dataset', 'dataset_revision', 'models', 'revisions', 'max_tokens'):
        if provenance.get(key) != metadata[key]:
            raise ValueError(f'Wrong {split} provenance: {key}')
    if (cache.get('source_model') != metadata['models']['llama']
            or cache.get('target_model') != metadata['models']['qwen']):
        raise ValueError(f'Wrong model identities: {split}')
    raw = Path(manifest_path).read_bytes()
    if hashlib.sha256(raw).hexdigest() != provenance['manifest_sha256']:
        raise ValueError(f'Manifest checksum mismatch: {split}')
    rows = [json.loads(line) for line in raw.splitlines()]
    ids = [row['id'] for row in rows]
    if (not rows or len(rows) != provenance['n_examples'] or cache['ids'] != ids
            or len(ids) != len(set(ids))):
        raise ValueError(f'Invalid row count/ID order: {split}')
    for row in rows:
        if row['split'] != split or not row['id'].startswith(split + '/'):
            raise ValueError(f'Invalid manifest ID/split: {split}')
        if not row['prefix'].strip() or not row['continuation'].strip():
            raise ValueError(f'Empty text: {row["id"]}')
        for name in widths:
            if not 0 < row[f'{name}_length'] <= metadata['max_tokens'] or row[f'{name}_target_length'] < 1:
                raise ValueError(f'Invalid token lengths: {row["id"]}')
    for name, width in widths.items():
        key = 'target_token_ids' if name == 'qwen' else 'llama_target_token_ids'
        labels = cache[key]
        expected = torch.tensor([row[f'{name}_target_id'] for row in rows], dtype=torch.int64)
        if (labels.dtype != torch.int64 or not torch.equal(labels, expected)
                or (labels < 0).any() or (labels >= vocab_sizes[name]).any()):
            raise ValueError(f'Invalid {name} label alignment/range: {split}')
        for suffix in ('', '_post'):
            matrix = cache[f'h_{name}{suffix}']
            if (matrix.shape != (len(rows), width) or not matrix.is_floating_point()
                    or str(matrix.dtype) != provenance['extraction']['compute_dtype']
                    or not torch.isfinite(matrix).all()):
                raise ValueError(f'Invalid {name}{suffix} matrix: {split}')
    return rows, cache


# Verified against both public parquet files at this immutable dataset revision.
# Each entry is [normalized-prefix SHA256, validation ID, test ID, full-text SHA256].
KNOWN_LAMBADA_EVAL_DUPLICATES = [
    [
        "8c7b48b64b8da574652312e119ea5da46177e4295117e2d349a5fd54efc81298",
        "validation/2060",
        "test/2754",
        "ab4179fbd5eba599bd3ee8b48398ebe05272ca3ce93a5c92ea244bc9f72fe3c4"
    ],
    [
        "a479fa89cf5052d12c41d9faeb1bb0d86893c5ee60fea8c08c27c5f912657cb1",
        "validation/66",
        "test/912",
        "7b48e6e1189837f234543b92fb2d2e2f3db2025939de1baa58764b55e442df2d"
    ],
    [
        "cafbc84c64f0fc3ecf9b6ef4bbf4de73a5d9a36138d25cee98fb78671a1294b0",
        "validation/1614",
        "test/2145",
        "6395acfbe10b2939197f818bfa1db55083d394a03d391e4ff83258ab7153fe88"
    ],
    [
        "b15a6d9bc279bec6ae3c49bd87431bae475a34a2647ceabc3cd3933cbd6f3dcd",
        "validation/1699",
        "test/1671",
        "e4288db3745fab74282606d2084c76d7124d4ca425fa2c38da41dbdf19b21be6"
    ],
    [
        "3cea84c1a8b8df3d48e0d10b3f00c61faf544bb1294ef59b225a90d75265e965",
        "validation/4233",
        "test/1781",
        "5bfbd9dcf79b620c04d403ca07673194dd87e561d3aec235bd73177f4a22693f"
    ],
    [
        "4560f9f30848aa75397832d6a3976ca2af4d02bd762e6e76e791778f0a06d137",
        "validation/3603",
        "test/1979",
        "b4e45e400f1f1999c57c561d0cb209fee63e3b29657eb8b45c73307549753e81"
    ],
    [
        "f3a41296db857ffa2f7f234a3e3bda929d0ba73c1c6ac3a6c6d2eca5cb344f11",
        "validation/745",
        "test/4356",
        "8cb9ab6b8641b5f76dde990dc6e4f1bc77d75519697c0886415b576e0184d8c2"
    ],
    [
        "98adc21f8abf7f7aad4d006b692e23589feb2eb2b11ea6f3386c265309955435",
        "validation/388",
        "test/3548",
        "7a75f253bb9abc97aed6b1f16eb39c82f1410658e1bb8c87557ee66b65dad764"
    ],
    [
        "482ff98a4cc02149f6ea70d4e3ce4eee124aa8416c32085f209330e824d264f7",
        "validation/4401",
        "test/1932",
        "d3c5381fd1aa50d4722df8b8d672fc7936690fca7b167b0dafc735b827e87712"
    ]
]


def audit_split_overlap(manifests, metadata):
    """Reject training leakage; register verified source duplicates as test exclusions."""
    from itertools import combinations
    by_split = {}
    for split, rows in manifests.items():
        index = {}
        for row in rows:
            index.setdefault(' '.join(row['prefix'].split()), []).append(row)
        by_split[split] = index
    verified_source = (metadata['dataset'] == 'cimec/lambada' and
        metadata['dataset_revision'] == '5953bd97664b64b95754f299b2309ecfbfbe81b9')
    known = {entry[0]: entry for entry in KNOWN_LAMBADA_EVAL_DUPLICATES}
    results, excluded_test_ids = [], set()
    for left, right in combinations(manifests, 2):
        left_ids = {r['id'] for r in manifests[left]}
        right_ids = {r['id'] for r in manifests[right]}
        shared_ids = left_ids & right_ids
        shared_contexts = set(by_split[left]) & set(by_split[right])
        if shared_ids:
            raise ValueError(f'Cross-split ID overlap: {left} vs. {right}; '
                             f'n={len(shared_ids)}, sample_ids={sorted(shared_ids)[:5]}')
        observed = []
        for context in sorted(shared_contexts):
            fingerprint = hashlib.sha256(context.encode()).hexdigest()
            entries = {left: by_split[left][context], right: by_split[right][context]}
            if 'train' in (left, right):
                raise ValueError(f'Training context overlaps {right if left == "train" else left}: '
                                 f'context_sha256={fingerprint}, '
                                 f'ids={ {k: [r["id"] for r in v] for k, v in entries.items()} }')
            expected = known.get(fingerprint)
            if not verified_source or set(entries) != {'validation', 'test'} or expected is None:
                raise ValueError(f'Unverified validation/test context overlap: {fingerprint}; '
                                 'inspect source manifests before using these artifacts')
            for split, expected_id in (('validation', expected[1]), ('test', expected[2])):
                rows = entries[split]
                if (len(rows) != 1 or rows[0]['id'] != expected_id or
                        hashlib.sha256((rows[0]['prefix'] + rows[0]['continuation']).encode()).hexdigest() != expected[3]):
                    raise ValueError(f'Overlap does not match pinned source evidence: {split}/{fingerprint}')
            excluded_test_ids.add(expected[2])
            observed.append(dict(context_sha256=fingerprint, validation_ids=[expected[1]],
                                 test_ids=[expected[2]], full_text_sha256=expected[3],
                                 verified_in_pinned_source=True))
        results.append(dict(left=left, right=right, shared_ids=len(shared_ids),
                            shared_contexts=len(shared_contexts), details=observed))
    test_ids = [row['id'] for row in manifests['test']]
    selected = [item for item in test_ids if item not in excluded_test_ids]
    if not selected:
        raise ValueError('No disjoint test examples remain')
    views = dict(
        test_disjoint=dict(n_examples=len(selected), excluded_ids=sorted(excluded_test_ids),
                           ids_sha256=object_hash(selected),
                           policy='Exclude test rows whose normalized prefixes occur in validation; train overlap is forbidden.'),
        test_official=dict(n_examples=len(test_ids), excluded_ids=[], ids_sha256=object_hash(test_ids),
                           policy='All accepted official test rows, including documented source duplicates.'),
    )
    return results, views



class PairedActivations(Dataset):
    """Keep saved precision on CPU; convert only requested rows to float32."""
    def __init__(self, cache, rows, indices=None):
        self.indices = tuple(range(len(rows))) if indices is None else tuple(indices)
        self.x, self.y = cache['h_llama'], cache['h_qwen']
        self.labels, self.ids = cache['target_token_ids'], tuple(cache['ids'])
        self.single = torch.tensor([r['qwen_target_length'] == 1 for r in rows])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        index = self.indices[index]
        return {'X': self.x[index].float(), 'Y': self.y[index].float(),
                'labels': self.labels[index], 'ids': self.ids[index],
                'single_token_mask': self.single[index]}


def make_loader(handoff, split, batch_size=32, seed=42, *, allow_test=False, test_view="disjoint"):
    """Only training shuffles. Test requires opt-in; disjoint is the primary test view."""
    if split not in ('train', 'validation', 'test') or batch_size <= 0:
        raise ValueError('Use a named split and a positive batch size')
    if split == 'test' and not allow_test:
        raise ValueError('Test is reserved for final evaluation; set allow_test=True explicitly')
    item = handoff['splits'][split]
    for suffix, field in (('cache', 'cache_path'), ('manifest', 'manifest_path')):
        artifact = handoff['artifacts'][f'{split}_{suffix}']
        if (Path(item[field]).resolve() != Path(artifact['path']).resolve()
                or file_hash(item[field]) != artifact['sha256']):
            raise ValueError(f'{split} {suffix} differs from the verified handoff')
    rows, cache = read_split(item['cache_path'], item['manifest_path'], split,
                            item['fingerprint'], handoff['metadata'],
                            handoff['widths'], handoff['vocab_sizes'])
    indices = None
    if split == 'test':
        if test_view not in ('disjoint', 'official'):
            raise ValueError('test_view must be disjoint or official')
        view = handoff['evaluation_views'][f'test_{test_view}']
        excluded = set(view['excluded_ids'])
        if not excluded.issubset(set(cache['ids'])):
            raise ValueError('Test view references IDs absent from this cache')
        indices = [i for i, value in enumerate(cache['ids']) if value not in excluded]
        selected_ids = [cache['ids'][i] for i in indices]
        if len(indices) != view['n_examples'] or object_hash(selected_ids) != view['ids_sha256']:
            raise ValueError('Test view count/order differs from the handoff')
    dataset = PairedActivations(cache, rows, indices)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=split == 'train',
                      generator=generator, num_workers=0, drop_last=False)


def new_stitch(source_width=3072, target_width=2048, seed=42):
    """Seeded PyTorch Linear initialization, with no change to the caller's RNG."""
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        return nn.Linear(source_width, target_width, bias=True, device='cpu', dtype=torch.float32)


def load_stitch(handoff, device='cpu'):
    item = handoff['artifacts']['affine_initialization']
    if file_hash(item['path']) != item['sha256']:
        raise ValueError('Affine initialization changed after handoff')
    checkpoint = torch.load(item['path'],
                            map_location='cpu', weights_only=True)
    if checkpoint['trained_steps'] != 0:
        raise ValueError('Expected an untrained initialization')
    config = handoff['affine']
    stitch = new_stitch(config['source_width'], config['target_width'], config['seed'])
    stitch.load_state_dict(checkpoint['state_dict'])
    return stitch.to(device)


class FrozenQwenHead(nn.Module):
    """Frozen buffers preserve exported precision while gradients flow to inputs."""
    def __init__(self, state):
        super().__init__()
        self.eps = state['norm_eps']
        self.register_buffer('norm_weight', state['norm_weight'])
        self.register_buffer('lm_head_weight', state['lm_head_weight'])

    def forward(self, hidden):
        hidden = hidden.to(self.norm_weight.dtype)
        normalized = hidden.float() * torch.rsqrt(hidden.float().square().mean(-1, keepdim=True) + self.eps)
        normalized = normalized.to(hidden.dtype) * self.norm_weight
        return F.linear(normalized, self.lm_head_weight)


def load_qwen_head(handoff, device='cpu'):
    item = handoff['artifacts']['qwen_head']
    if file_hash(item['path']) != item['sha256']:
        raise ValueError('Qwen head changed after handoff')
    state = torch.load(item['path'], map_location='cpu', weights_only=True)
    if (state['provenance']['run_hash'] != handoff['run_hash']
            or state['provenance']['model'] != handoff['metadata']['models']['qwen']
            or state['provenance']['revision'] != handoff['metadata']['revisions']['qwen']):
        raise ValueError('Qwen head provenance differs from the handoff')
    return FrozenQwenHead(state).eval().requires_grad_(False).to(device)


def objective_loss(stitch, batch, objective, head=None, temperature=1.0):
    """One objective per experiment; calling this does not perform an optimizer step."""
    if objective not in ('mse', 'ce', 'kl'):
        raise ValueError('Select mse, ce, or kl')
    device = next(stitch.parameters()).device
    predicted = stitch(batch['X'].to(device=device, dtype=torch.float32))
    if objective == 'mse':
        return F.mse_loss(predicted, batch['Y'].to(device=device, dtype=torch.float32))
    if head is None:
        raise ValueError('CE/KL require the exported frozen Qwen head on the stitch device')
    if any(p.requires_grad for p in head.parameters()):
        raise ValueError('The output head must be frozen')
    # Keep this student forward differentiable, including the cast to saved head precision.
    logits = head(predicted).float()
    if objective == 'ce':
        return F.cross_entropy(logits, batch['labels'].to(device))
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError('KL temperature must be positive and finite')
    with torch.no_grad():
        teacher_logits = head(batch['Y'].to(device=device, dtype=torch.float32)).float()
        teacher_logp = F.log_softmax(teacher_logits / temperature, dim=-1)
    student_logp = F.log_softmax(logits / temperature, dim=-1)
    # KL(native Qwen || stitched Qwen), mean across examples, summed across vocabulary.
    return F.kl_div(student_logp, teacher_logp, log_target=True, reduction='batchmean') * temperature**2

# Running Phase 5 verifies artifacts and exports files; it does not train or score test data.
import gc
import json
import os
from pathlib import Path
from tempfile import NamedTemporaryFile

import torch

HANDOFF_SEED = 42
HANDOFF_BATCH_SIZE = 32

required_p5 = ('CACHE_RUN_DIR', 'TRAIN_DIR', 'training_meta', 'HIDDEN_SIZES',
               'show_table', 'verify_exported_head', 'baseline_file_hash', 'baseline_object_hash')
missing_p5 = [name for name in required_p5 if name not in globals()]
if missing_p5:
    raise RuntimeError(f'Run configuration and Phases 1–3 first. Missing: {missing_p5}')

root = Path(CACHE_RUN_DIR).resolve()
run = json.loads((root / 'run.json').read_text())
completed = json.loads((root / 'completed.json').read_text())
if completed.get('identity') != run['identity']:
    raise ValueError('The selected parent extraction run is incomplete')
if json.loads((Path(TRAIN_DIR) / 'provenance.json').read_text()) != training_meta:
    raise ValueError('Training provenance changed since Phase 1')
if file_hash(Path(TRAIN_DIR) / 'examples.jsonl') != training_meta['manifest_sha256']:
    raise ValueError('Training source manifest changed since Phase 1')
run_hash_p5 = object_hash(run['identity'])
for key in ('models', 'revisions'):
    if run['identity'][key] != training_meta[key]:
        raise ValueError(f'Parent run differs from selected training data: {key}')

# Require the complete revised Phase 4 report, including the live parity checks.
p4_path = root / 'reports/phase4_validation_report.json'
p4 = json.loads(p4_path.read_text())
checks = p4.get('checks', {})
expected_checks = {'Split separation (train vs. validation)', 'Train-only constant mean provenance'}
for split in ('train', 'validation'):
    expected_checks.update({f'Data & label alignment ({split})', f'Tensor & disk-reload integrity ({split})',
                            f'Live tokenizer boundary audit ({split})'})
for name in ('llama', 'qwen'):
    expected_checks.update({f'Pre->post RMSNorm ({name}, {split})' for split in ('train', 'validation')})
    expected_checks.update({f'Batched vs. single activation ({name})',
                            f'Frozen-head logit reconstruction ({name})',
                            f'End-to-end cached->native logits ({name})'})
    metric_names = [key for key in checks if key.startswith(f'Independent metric recomputation ({name}, n=')]
    if len(metric_names) != 1:
        raise ValueError(f'Missing independent {name} metric check in Phase 4')
    expected_checks.update(metric_names)
    evidence = checks.get(f'Batched vs. single activation ({name})', {}).get('per_example_errors', {})
    for group in ('batch_vs_single', 'cache_vs_single'):
        if not evidence.get(group) or any(not item.get('passed') for item in evidence[group]):
            raise ValueError('Run the revised Phase 4 activation magnitude/parity checks first')
p4_issues = []
if Path(p4.get('artifact_dir', '')).resolve() != root:
    p4_issues.append('artifact directory differs')
if p4.get('analysis_splits') != ['train', 'validation']:
    p4_issues.append('expected train and validation analysis splits')
if p4.get('run_live_model_checks') is not True:
    p4_issues.append('live model checks were not enabled')
if expected_checks - checks.keys():
    p4_issues.append(f"missing checks: {sorted(expected_checks - checks.keys())}")
failed_checks = [key for key, item in checks.items() if item.get('passed') is not True]
if failed_checks:
    p4_issues.append(f'failed checks: {failed_checks}')
if p4_issues:
    raise ValueError('Phase 5 requires a complete, passing Phase 4 report. ' + '; '.join(p4_issues))

# Select only a completed full test extraction belonging to this train/validation run.
selected_test = globals().get('TEST_CACHE_RUN_DIR')
if selected_test is None:
    candidates = []
    for candidate in sorted((root / 'test_runs').glob('*/completed.json')):
        item = json.loads(candidate.read_text())
        if item.get('parent_run_hash') == run_hash_p5:
            candidates.append(candidate.parent)
    if len(candidates) != 1:
        raise ValueError(f'Found {len(candidates)} completed test runs. Set TEST_CACHE_RUN_DIR '
                         'in configuration to the directory printed by activation_test.ipynb.')
    selected_test = candidates[0]
TEST_CACHE_RUN_DIR = Path(selected_test).resolve()
test_run = json.loads((TEST_CACHE_RUN_DIR / 'run.json').read_text())
test_done = json.loads((TEST_CACHE_RUN_DIR / 'completed.json').read_text())
if (test_run.get('parent_run_hash') != run_hash_p5 or test_done.get('parent_run_hash') != run_hash_p5
        or test_done.get('identity') != test_run['identity']
        or set(test_run['identity']['split_fingerprints']) != {'test'}):
    raise ValueError('Test run must be complete and reference the selected parent run')

for key in ('models', 'revisions'):
    if test_run['identity'][key] != run['identity'][key]:
        raise ValueError(f'Test run differs from parent: {key}')
artifacts, vocab_sizes, head_provenance = {}, {}, {}


def record_artifact(key, path):
    path = Path(path).resolve()
    artifacts[key] = dict(path=str(path), sha256=file_hash(path), bytes=path.stat().st_size)
    return artifacts[key]


for name in ('llama', 'qwen'):
    state, _ = verify_exported_head(root, name, training_meta['models'][name],
        training_meta['revisions'][name], run_hash_p5, HIDDEN_SIZES[name])
    vocab_sizes[name] = state['vocab_size']
    head_provenance[name] = state['provenance']
    record_artifact(f'{name}_head', root / f'{name}_frozen_head.pt')
    record_artifact(f'{name}_head_checksum', root / f'{name}_frozen_head.sha256.json')
    record_artifact(f'{name}_head_validation', root / f'{name}_head_validation.json')
    del state
    gc.collect()

splits, handoff_manifests = {}, {}
for split, directory, identity in (
    ('train', root, run['identity']), ('validation', root, run['identity']),
    ('test', TEST_CACHE_RUN_DIR, test_run['identity']),
):
    folder = directory / split
    fingerprint = identity['split_fingerprints'][split]
    rows, cache = read_split(folder / 'paired.pt', folder / 'examples.jsonl',
        split, fingerprint, training_meta, HIDDEN_SIZES, vocab_sizes)
    if split == 'test':
        meta = cache['provenance']
        accounting = meta['accounting']
        if (meta.get('parent_run_hash') != run_hash_p5 or accounting.get('requested') is not None
                or accounting['examined'] != accounting['total_source_rows']
                or accounting['accepted'] != len(rows)
                or accounting['examined'] != len(rows) + sum(accounting['rejected'].values())):
            raise ValueError('Select the full accepted test split, with complete preprocessing accounting')
        if any(meta['extraction'][key] != run['settings'][key] for key in ('compute_dtype', 'attention')):
            raise ValueError('Test precision/attention differs from parent extraction')
        for name in ('llama', 'qwen'):
            ref = meta['head_references'][name]
            if (ref['sha256'] != artifacts[f'{name}_head']['sha256']
                    or ref['provenance'] != head_provenance[name]):
                raise ValueError(f'Test references different {name} output components')
    handoff_manifests[split] = rows
    splits[split] = dict(cache_path=str(folder / 'paired.pt'), manifest_path=str(folder / 'examples.jsonl'),
                         fingerprint=fingerprint, n_examples=len(rows),
                         stored_dtype=str(cache['h_qwen'].dtype),
                         manifest_sha256=cache['provenance']['manifest_sha256'])
    for suffix, filename in (('cache', 'paired.pt'), ('checksum', 'paired.sha256.json'),
                             ('manifest', 'examples.jsonl')):
        record_artifact(f'{split}_{suffix}', folder / filename)
    del rows, cache
    gc.collect()

overlap_audit, evaluation_views = audit_split_overlap(handoff_manifests, training_meta)

metrics_path_p5 = root / 'baselines/metrics.json'
metrics_p5 = json.loads(metrics_path_p5.read_text())
mean_path_p5 = root / 'baselines/qwen_training_mean.pt'
if file_hash(mean_path_p5) != metrics_p5['training_mean_sha256']:
    raise ValueError('Baseline training mean changed since metrics were computed')
expected_baselines = {(split, name) for split in ('train', 'validation')
                      for name in ('native_llama', 'native_qwen', 'constant_qwen_train_mean')}
if ({(r['split'], r['baseline']) for r in metrics_p5['results']} != expected_baselines
        or len(metrics_p5['results']) != len(expected_baselines)):
    raise ValueError('Expected exactly the train/validation native and constant baseline results')
for row in metrics_p5['results']:
    if (row['n_examples'] != splits[row['split']]['n_examples']
            or row['manifest_sha256'] != splits[row['split']]['manifest_sha256']):
        raise ValueError('Baseline metrics refer to different example manifests')
for name in ('llama', 'qwen'):
    if (metrics_p5['heads'][name]['checksum'] != artifacts[f'{name}_head']['sha256']
            or metrics_p5['heads'][name]['provenance'] != head_provenance[name]):
        raise ValueError('Baseline output components differ from the handoff')
for key, path in (
    ('training_provenance', Path(TRAIN_DIR) / 'provenance.json'),
    ('training_source_manifest', Path(TRAIN_DIR) / 'examples.jsonl'),
    ('parent_run', root / 'run.json'), ('parent_completed', root / 'completed.json'),
    ('test_run', TEST_CACHE_RUN_DIR / 'run.json'), ('test_completed', TEST_CACHE_RUN_DIR / 'completed.json'),
    ('baseline_metrics', metrics_path_p5), ('training_mean', mean_path_p5), ('phase4_validation', p4_path),
):
    record_artifact(key, path)


def atomic_export(path, payload, tensor=False):
    with NamedTemporaryFile(dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        if tensor:
            torch.save(payload, temporary)
        else:
            temporary.write_text(payload, encoding='utf-8')
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


handoff_dir = root / 'handoff'
handoff_dir.mkdir(parents=True, exist_ok=True)
affine_config = dict(source_width=HIDDEN_SIZES['llama'], target_width=HIDDEN_SIZES['qwen'],
                     bias=True, dtype='torch.float32', seed=HANDOFF_SEED,
                     initialization='PyTorch nn.Linear default; shared checkpoint across objectives')
untrained_stitch = new_stitch(affine_config['source_width'], affine_config['target_width'], HANDOFF_SEED)
initial_path = handoff_dir / 'affine_init.pt'
atomic_export(initial_path, dict(state_dict=untrained_stitch.state_dict(), config=affine_config,
                               trained_steps=0, torch_version=str(torch.__version__)), tensor=True)
record_artifact('affine_initialization', initial_path)
handoff_report = dict(
    schema='cross-family-milestone2-v2', task='first continuation token',
    interface_definition='Functions defined directly in milestone1.ipynb, Phase 5',
    metadata=training_meta, widths=HIDDEN_SIZES, vocab_sizes=vocab_sizes, run_hash=run_hash_p5,
    parent_cache_run_dir=str(root), test_cache_run_dir=str(TEST_CACHE_RUN_DIR),
    splits=splits, affine=affine_config, artifacts=artifacts,
    overlap_audit=overlap_audit, evaluation_views=evaluation_views,
    extraction_environment=run['environment'], interface_runtime={'torch': str(torch.__version__)},
    baseline_results=metrics_p5['results'],
    validation=dict(passed=True, checks=len(checks), report=str(p4_path)),
    objectives={
        'mse': 'Mean squared error across examples and Qwen pre-RMSNorm coordinates.',
        'ce': 'Mean cross-entropy over examples, using Qwen first-continuation-token labels.',
        'kl': 'KL(native Qwen || stitched Qwen), sum over vocabulary / batch size; T=1 by default, T^2 scaling.',
    },
    protocol=dict(training='Shuffle train only; no dropped rows. Use the same saved initialization across objectives.',
                  selection='Validation selects hyperparameters and checkpoints.',
                  test='Require allow_test=True. Default disjoint view excludes documented validation duplicates; test_view=official preserves all accepted rows. Reserve both scores for final evaluation.',
                  precision='Float32 stitch and X/Y batches; frozen head uses exported precision; losses use float32.',
                  gradients='Only affine weights/bias train. Student head forward tracks gradients; native teacher uses no_grad.'),
)
HANDOFF_PATH = handoff_dir / 'handoff.json'
atomic_export(HANDOFF_PATH, json.dumps(handoff_report, indent=2, allow_nan=False) + '\n')
# Reuse these functions in Milestone 2; load_handoff verifies every recorded artifact.
verified_handoff = load_handoff(HANDOFF_PATH)
train_loader = make_loader(verified_handoff, 'train', HANDOFF_BATCH_SIZE, HANDOFF_SEED)
validation_loader = make_loader(verified_handoff, 'validation', HANDOFF_BATCH_SIZE, HANDOFF_SEED)
stitch = load_stitch(verified_handoff)
assert len(train_loader.dataset) == splits['train']['n_examples']
assert len(validation_loader.dataset) == splits['validation']['n_examples']
assert all(torch.equal(stitch.state_dict()[key], value) for key, value in untrained_stitch.state_dict().items())
del untrained_stitch

show_table('Milestone 2 artifact handoff', ['Split', 'Examples', 'Stored dtype', 'Cache path'],
           [[split, item['n_examples'], item['stored_dtype'], item['cache_path']] for split, item in splits.items()])
show_table('Final test evaluation views', ['View', 'Examples', 'Excluded test IDs'],
           [[name, item['n_examples'], ', '.join(item['excluded_ids']) or '—']
            for name, item in evaluation_views.items()])
show_table('Objective interfaces', ['Objective', 'Definition'], list(handoff_report['objectives'].items()))
show_table('Handoff verification', ['Measure', 'Value'], [
    ['Verified artifacts', len(artifacts)], ['Passing Phase 4 checks', len(checks)],
    ['Affine dimensions', f"{affine_config['source_width']} → {affine_config['target_width']} (with bias)"],
    ['Training steps', 0], ['Test metrics computed', 0],
])
print('HANDOFF_PATH =', repr(str(HANDOFF_PATH)))
print('Shared untrained initialization:', initial_path)
print('Prepared train_loader, validation_loader, and stitch on CPU. No optimizer steps were run.')


### Milestone 2 artifact handoff

| Split | Examples | Stored dtype | Cache path |
| --- | --- | --- | --- |
| train | 10000 | torch.bfloat16 | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train/paired.pt |
| validation | 4869 | torch.bfloat16 | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation/paired.pt |
| test | 5153 | torch.bfloat16 | /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/test_runs/test5153_d75faa265fa6/test/paired.pt |

### Final test evaluation views

| View | Examples | Excluded test IDs |
| --- | --- | --- |
| test_disjoint | 5144 | test/1671, test/1781, test/1932, test/1979, test/2145, test/2754, test/3548, test/4356, test/912 |
| test_official | 5153 | — |

### Objective interfaces

| Objective | Definition |
| --- | --- |
| mse | Mean squared error across examples and Qwen pre-RMSNorm coordinates. |
| ce | Mean cross-entropy over examples, using Qwen first-continuation-token labels. |
| kl | KL(native Qwen \|\| stitched Qwen), sum over vocabulary / batch size; T=1 by default, T^2 scaling. |

### Handoff verification

| Measure | Value |
| --- | --- |
| Verified artifacts | 25 |
| Passing Phase 4 checks | 20 |
| Affine dimensions | 3072 → 2048 (with bias) |
| Training steps | 0 |
| Test metrics computed | 0 |

HANDOFF_PATH = '/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/handoff/handoff.json'
Shared untrained initialization: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/handoff/affine_init.pt
Prepared train_loader, validation_loader, and stitch on CPU. No optimizer steps were run.
